In [204]:
from jax import numpy as jnp
from flax import nnx
import optax
from tqdm import tqdm

In [ ]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
import jax_dataloader as jdl

class FashionMNISTCSV(Dataset):
    def __init__(self, path: str):
        df = pd.read_csv(path, compression='gzip')
        self.labels = df.iloc[:, 0].values
        self.images = df.iloc[:, 1:].values.astype(np.float32) / 255.0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

ds_train = FashionMNISTCSV("fashion-mnist_train.csv.gz")
ds_val   = FashionMNISTCSV("fashion-mnist_test.csv.gz")

dl_train = jdl.DataLoader(ds_train, 'pytorch', batch_size=1024, shuffle=True)
dl_val   = jdl.DataLoader(ds_val,   'pytorch', batch_size=1024, shuffle=False)

In [206]:
for item in dl_train:
    print(item[1])
    break

[7 0 9 ... 5 4 6]


In [207]:
class MLP(nnx.Module):
    def __init__(self, in_features: int, out_features: int, rngs: nnx.Rngs):
        self.l1 = nnx.Linear(in_features, 512, rngs=rngs)
        self.l2 = nnx.Linear(512, 512, rngs=rngs)
        self.l3 = nnx.Linear(512, 512, rngs=rngs)
        self.l4 = nnx.Linear(512, out_features, rngs=rngs)
        self.dropout = nnx.Dropout(0.5, rngs=rngs)

    def __call__(self, X: jnp.ndarray):
        X = X.reshape((X.shape[0], -1))
        X = nnx.relu(self.dropout(self.l1(X)))
        X = nnx.relu(self.dropout(self.l2(X)))
        X = nnx.relu(self.dropout(self.l3(X)))
        return self.l4(X)

In [208]:
rngs = nnx.Rngs(0)
model = MLP(784, 10, rngs)

tx = optax.chain(optax.add_decayed_weights(1e-4), optax.sgd(learning_rate=0.03))
# tx = optax.sgd(learning_rate=0.03)
optimiser = nnx.Optimizer(model, tx, wrt=nnx.Param)

class MLP_Metrics(nnx.metrics.MultiMetric):
    loss: nnx.metrics.Average
    acc: nnx.metrics.Accuracy

metrics = MLP_Metrics(loss=nnx.metrics.Average(), acc=nnx.metrics.Accuracy())

def forward(model: MLP, metrics: nnx.Metric, X: jnp.ndarray, y: jnp.ndarray):
        y_hat = model(X)
        loss = optax.softmax_cross_entropy_with_integer_labels(y_hat, y).mean()
        metrics.update(values=loss, logits=y_hat, labels=y)
        return loss

@nnx.jit
def train_step(model: MLP, optimiser: nnx.Optimizer, metrics: nnx.Metric, X: jnp.ndarray, y: jnp.ndarray):
    print("compile")
    loss, grads = nnx.value_and_grad(forward)(model, metrics, X, y)
    optimiser.update(model, grads)
    return loss

@nnx.jit
def eval_step(model: MLP, metrics: nnx.Metric, X: jnp.ndarray, y: jnp.ndarray):
    forward(model, metrics, X, y)

In [201]:
data = {"train_n": [], "val_n": [], "train_r": [], "val_r": []}

In [209]:
epochs = 100

for e in tqdm(range(epochs)):
    model.train()
    for batch in dl_train:
        loss = train_step(model, optimiser, metrics, batch[0], batch[1])
    data["train_r"].append(metrics.compute()["loss"])
    metrics.reset()

    model.eval()
    for batch in dl_val:
        loss = eval_step(model, metrics, batch[0], batch[1])
    data["val_r"].append(metrics.compute()["loss"])
    metrics.reset()

  0%|          | 0/100 [00:00<?, ?it/s]

compile
compile


100%|██████████| 100/100 [00:43<00:00,  2.32it/s]


In [210]:
data

{'train_n': [Array(1.6753116, dtype=float32),
  Array(0.94032735, dtype=float32),
  Array(0.77256477, dtype=float32),
  Array(0.71511745, dtype=float32),
  Array(0.65146023, dtype=float32),
  Array(0.6170167, dtype=float32),
  Array(0.5865324, dtype=float32),
  Array(0.5605099, dtype=float32),
  Array(0.5521555, dtype=float32),
  Array(0.5284215, dtype=float32),
  Array(0.5088255, dtype=float32),
  Array(0.4993107, dtype=float32),
  Array(0.48499325, dtype=float32),
  Array(0.492548, dtype=float32),
  Array(0.48013526, dtype=float32),
  Array(0.46403956, dtype=float32),
  Array(0.4594226, dtype=float32),
  Array(0.4571712, dtype=float32),
  Array(0.44762805, dtype=float32),
  Array(0.44092238, dtype=float32),
  Array(0.43410733, dtype=float32),
  Array(0.44243324, dtype=float32),
  Array(0.43172768, dtype=float32),
  Array(0.42252472, dtype=float32),
  Array(0.42316937, dtype=float32),
  Array(0.42092812, dtype=float32),
  Array(0.4095926, dtype=float32),
  Array(0.41323695, dtype=floa

In [211]:
import plotly.express as px
print(data)
px.line(data)

{'train_n': [Array(1.6753116, dtype=float32), Array(0.94032735, dtype=float32), Array(0.77256477, dtype=float32), Array(0.71511745, dtype=float32), Array(0.65146023, dtype=float32), Array(0.6170167, dtype=float32), Array(0.5865324, dtype=float32), Array(0.5605099, dtype=float32), Array(0.5521555, dtype=float32), Array(0.5284215, dtype=float32), Array(0.5088255, dtype=float32), Array(0.4993107, dtype=float32), Array(0.48499325, dtype=float32), Array(0.492548, dtype=float32), Array(0.48013526, dtype=float32), Array(0.46403956, dtype=float32), Array(0.4594226, dtype=float32), Array(0.4571712, dtype=float32), Array(0.44762805, dtype=float32), Array(0.44092238, dtype=float32), Array(0.43410733, dtype=float32), Array(0.44243324, dtype=float32), Array(0.43172768, dtype=float32), Array(0.42252472, dtype=float32), Array(0.42316937, dtype=float32), Array(0.42092812, dtype=float32), Array(0.4095926, dtype=float32), Array(0.41323695, dtype=float32), Array(0.40366396, dtype=float32), Array(0.405655